# Official-core Informer ProbSparse Classifier V2 for UCR

This notebook adapts the **official Informer encoder core** to UCR time-series
classification while following the completed `train_all_twotower_v2.ipynb`
experiment protocol.

The model-specific Informer components are:

- circular-convolution value embedding plus sinusoidal positional embedding;
- ProbSparse self-attention (`factor=5`) in every encoder layer;
- residual attention and 1x1-convolution feed-forward blocks;
- official-style Conv1d–BatchNorm–ELU–MaxPool distillation between layers;
- temporal mean pooling, dropout, and a classification head.

The forecasting-only decoder and calendar/time-marker embedding are intentionally
omitted because UCR supplies a single univariate sequence and one class label per
sample, not future decoder targets or calendar covariates. This is therefore an
**official-core task adaptation**, not a claim that the unmodified forecasting
script itself performs classification.

The surrounding experiment protocol is identical to Two-Tower V2:

- deterministic shared train/validation indices from the official TRAIN split;
- official TEST is never used for checkpoint selection;
- per-sample time-series standardization and train-derived sequence length;
- binary-column / one-hot labels with `BCEWithLogitsLoss`;
- corrected binary ACC/AUC and identical multiclass metrics;
- Adam (`1e-4`), batch size 8, cosine scheduler, seed 42, and 60 full epochs;
- best checkpoint selected by minimum validation BCE loss;
- official TEST evaluated exactly once after restoring that checkpoint;
- resumable isolated output under `final_runs_v2/informer_probsparse/seed_42`.

Dataset discovery deliberately follows the same marker-file rule as Two-Tower V2
so the same 125 datasets are requested. Informer never reads or uses Aout values.
Any dataset that cannot complete the common protocol is recorded generically in
`failed_datasets.csv`.

## First local run

1. Confirm `DATA_ROOT` and `OUTPUT_ROOT` in Cell 1.
2. Leave `RUN_SMOKE_TEST=True` and `RUN_FULL_EXPERIMENT=False`, restart the kernel,
   clear outputs, and run all cells.
3. Confirm Coffee and ArrowHead complete under
   `_debug/informer_probsparse/seed_42`; the audit must report
   `uses_aout=False`, `attention=ProbSparse`, and `distil=True`.
4. Set `RUN_SMOKE_TEST=False`, `RUN_FULL_EXPERIMENT=True`, restart the kernel,
   clear outputs, and run all cells for the final 125-request experiment.


In [ ]:
# Cell 1 - imports and experiment configuration
from __future__ import annotations

import copy
import json
import os
import platform
import random
import sys
import traceback
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.backends.cudnn as cudnn
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset


# -----------------------------------------------------------------------------
# EDIT THESE TWO PATHS FOR YOUR WINDOWS MACHINE
# -----------------------------------------------------------------------------
DATA_ROOT = Path(r"D:\2025暑期科研\UCRArchive_2018\UCRArchive_2018")
OUTPUT_ROOT = Path(r"D:\2025暑期科研\UCRArchive_2018\TwoTower_Project_V2\final_runs_v2")


# Shared V2 experiment settings
MODEL_NAME = "informer_probsparse_lightweight"
EXPERIMENT_VERSION = "informer_probsparse_lightweight_ucr_v2"
SEED = 42
BATCH_SIZE = 8
NUM_EPOCHS = 60
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
T_MAX = 50
ETA_MIN = 0.0
VAL_FRACTION = 0.20
MIN_EFFECTIVE_LENGTH = 8
NUM_WORKERS = 0

# Official-core Informer architecture adapted to UCR classification
INFORMER_D_MODEL = 64          # alter: 32, 128
INFORMER_N_HEADS = 2           # alter: 1, 4
INFORMER_FACTOR = 3            # alter: 5
INFORMER_E_LAYERS = 2          # alter: 1, 3
INFORMER_D_FF = 128            # alter: 64, 256
INFORMER_DROPOUT = 0.1         # alter: 0.2, 0.3
INFORMER_ATTN_DROPOUT = 0.1    # alter: 0.2
INFORMER_DISTIL = True
INFORMER_ACTIVATION = "gelu"
INFORMER_POOL = "mean"

# Train all 60 epochs for equal-length Figure 2/3 histories. The final TEST
# metrics still use the checkpoint with the minimum validation BCE loss.
TRAIN_FULL_EPOCHS_FOR_CURVES = True

# Safe switches: smoke and final outputs are stored separately.
RUN_SMOKE_TEST = False
RUN_FULL_EXPERIMENT = False
SMOKE_DATASETS = ["Coffee", "ArrowHead"]  # binary + multiclass
SMOKE_EPOCHS = 3

RESUME_COMPLETED_DATASETS = True
SAVE_INDIVIDUAL_CURVES = True
DEVICE_REQUEST = "cuda:0"

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Cell 2 - determinism, paths, atomic persistence, and run metadata
def set_global_seed(seed: int) -> None:
    """Reset all relevant RNGs. Called again at the start of every dataset."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False


set_global_seed(SEED)


def resolve_device(requested: str) -> torch.device:
    if requested.startswith("cuda") and torch.cuda.is_available():
        return torch.device(requested)
    return torch.device("cpu")


DEVICE = resolve_device(DEVICE_REQUEST)
print("Resolved device:", DEVICE)


def model_output_dir(debug: bool = False) -> Path:
    if debug:
        return OUTPUT_ROOT / "_debug" / MODEL_NAME / f"seed_{SEED}"
    return OUTPUT_ROOT / MODEL_NAME / f"seed_{SEED}"


def ensure_output_tree(base_dir: Path) -> dict[str, Path]:
    paths = {
        "base": base_dir,
        "checkpoints": base_dir / "checkpoints",
        "curves": base_dir / "curves",
        "dataset_histories": base_dir / "dataset_histories",
        "shared_splits": OUTPUT_ROOT / "_shared_splits" / f"seed_{SEED}",
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths


def atomic_write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False, encoding="utf-8")
    os.replace(tmp, path)


def atomic_write_json(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def atomic_torch_save(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)


def load_own_checkpoint(path: Path, device: torch.device) -> dict:
    """Load this notebook's own tensor/primitives-only checkpoint safely."""
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:  # compatibility with older PyTorch
        return torch.load(path, map_location=device)


def append_or_replace_dataset_rows(
    csv_path: Path,
    new_df: pd.DataFrame,
    dataset_name: str,
) -> pd.DataFrame:
    if csv_path.exists():
        old_df = pd.read_csv(csv_path)
        if "dataset" in old_df.columns:
            old_df = old_df[old_df["dataset"] != dataset_name]
        combined = pd.concat([old_df, new_df], ignore_index=True)
    else:
        combined = new_df.copy()
    sort_cols = [column for column in ["dataset", "epoch"] if column in combined.columns]
    if sort_cols:
        combined = combined.sort_values(sort_cols).reset_index(drop=True)
    atomic_write_csv(combined, csv_path)
    return combined


def base_run_config(debug: bool, epochs: int) -> dict:
    return {
        "experiment_version": EXPERIMENT_VERSION,
        "model": MODEL_NAME,
        "model_class": "InformerProbSparseClassifier",
        "informer_implementation": "official-core ProbSparse encoder adapted to UCR classification",
        "official_reference": "https://github.com/zhouhaoyi/Informer2020",
        "attention": "ProbSparse",
        "uses_aout": False,
        "seed": SEED,
        "debug": bool(debug),
        "data_root": str(DATA_ROOT),
        "output_root": str(OUTPUT_ROOT),
        "device_requested": DEVICE_REQUEST,
        "device_resolved": str(DEVICE),
        "batch_size": BATCH_SIZE,
        "num_epochs": int(epochs),
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "scheduler": "CosineAnnealingLR",
        "t_max": T_MAX,
        "eta_min": ETA_MIN,
        "val_fraction": VAL_FRACTION,
        "d_model": INFORMER_D_MODEL,
        "n_heads": INFORMER_N_HEADS,
        "probsparse_factor": INFORMER_FACTOR,
        "encoder_layers": INFORMER_E_LAYERS,
        "d_ff": INFORMER_D_FF,
        "dropout": INFORMER_DROPOUT,
        "attention_dropout": INFORMER_ATTN_DROPOUT,
        "distillation": INFORMER_DISTIL,
        "activation": INFORMER_ACTIVATION,
        "pooling": INFORMER_POOL,
        "label_encoding": "one_hot_or_binary_column",
        "loss": "BCEWithLogitsLoss",
        "binary_prediction": "sigmoid(logit) >= 0.5",
        "multiclass_prediction": "argmax(logits)",
        "multiclass_auc": "macro one-vs-rest over valid classes",
        "checkpoint_selection": "minimum validation BCE loss",
        "test_evaluation": "once after restoring best checkpoint",
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }


In [ ]:
# Cell 3 - preprocessing and deterministic shared train/validation split
def clean_and_pad_timeseries(
    raw_2d: np.ndarray,
    min_len: int = 8,
    cap_len: int | None = None,
    pad_value: float = 0.0,
    per_sample_standardize: bool = True,
    fixed_len: int | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Clean tail NaNs, optionally z-score each sample, then pad or truncate."""
    rows, keep_idx, real_lens = [], [], []

    for i, row in enumerate(raw_2d):
        valid_vals = row[~np.isnan(row)]
        length = int(valid_vals.shape[0])
        if length < min_len:
            continue
        if per_sample_standardize:
            mu = float(valid_vals.mean())
            sigma = float(valid_vals.std())
            valid_vals = (valid_vals - mu) / (sigma if sigma > 0 else 1.0)
        rows.append(valid_vals)
        keep_idx.append(i)
        real_lens.append(length)

    if not rows:
        raise ValueError("All samples were filtered out; lower min_len or inspect the data.")

    max_real_len = max(real_lens)
    if fixed_len is not None:
        target_len = int(fixed_len)
    elif cap_len is not None:
        target_len = min(max_real_len, int(cap_len))
    else:
        target_len = max_real_len

    padded = []
    for values in rows:
        if values.shape[0] >= target_len:
            values = values[:target_len]
        else:
            values = np.pad(
                values,
                (0, target_len - values.shape[0]),
                constant_values=pad_value,
            )
        padded.append(values)

    return (
        np.stack(padded, axis=0).astype("float32"),
        np.asarray(keep_idx, dtype=np.int64),
        np.asarray(real_lens, dtype=np.int64),
    )


def make_class_safe_split(
    labels: np.ndarray,
    dataset_name: str,
    split_dir: Path,
    val_fraction: float = 0.20,
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray, Path]:
    """
    Create or reload a deterministic split from the cleaned official TRAIN set.

    Each class keeps at least one sample in the optimization subset. Classes with
    only one sample remain entirely in training. The saved split is shared by all
    V2 models and validated before reuse.
    """
    split_dir.mkdir(parents=True, exist_ok=True)
    split_path = split_dir / f"{dataset_name}_split.npz"
    labels = np.asarray(labels)

    if split_path.exists():
        saved = np.load(split_path, allow_pickle=False)
        train_idx = saved["train_idx"].astype(np.int64)
        val_idx = saved["val_idx"].astype(np.int64)
        saved_n = int(saved["n_samples"])
        if saved_n != len(labels):
            raise ValueError(
                f"Saved split for {dataset_name} has n={saved_n}, "
                f"but current cleaned training data has n={len(labels)}. "
                "Delete only this split file after verifying preprocessing."
            )
        return train_idx, val_idx, split_path

    rng = np.random.default_rng(seed)
    train_parts, val_parts = [], []
    for class_value in np.unique(labels):
        class_idx = np.flatnonzero(labels == class_value)
        class_idx = rng.permutation(class_idx)
        if len(class_idx) <= 1:
            n_val = 0
        else:
            n_val = max(1, int(round(len(class_idx) * val_fraction)))
            n_val = min(n_val, len(class_idx) - 1)
        val_parts.append(class_idx[:n_val])
        train_parts.append(class_idx[n_val:])

    train_idx = np.sort(np.concatenate(train_parts)).astype(np.int64)
    nonempty_val_parts = [part for part in val_parts if len(part) > 0]
    if nonempty_val_parts:
        val_idx = np.sort(np.concatenate(nonempty_val_parts)).astype(np.int64)
    else:
        raise ValueError(
            f"{dataset_name} cannot create a non-empty class-safe validation split."
        )

    if set(train_idx).intersection(set(val_idx)):
        raise RuntimeError("Train/validation indices overlap.")
    if len(train_idx) + len(val_idx) != len(labels):
        raise RuntimeError("Train/validation split does not cover every cleaned sample.")

    np.savez_compressed(
        split_path,
        train_idx=train_idx,
        val_idx=val_idx,
        n_samples=np.asarray(len(labels), dtype=np.int64),
        seed=np.asarray(seed, dtype=np.int64),
    )
    return train_idx, val_idx, split_path


In [ ]:
# Cell 4 - official-core Informer encoder adapted to UCR classification
import math


class SinusoidalPositionalEmbedding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 10000):
        super().__init__()
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        table = torch.zeros(max_len, d_model, dtype=torch.float32)
        table[:, 0::2] = torch.sin(position * div_term)
        table[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("table", table.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.size(1) > self.table.size(1):
            raise ValueError(
                f"Sequence length {x.size(1)} exceeds positional table "
                f"length {self.table.size(1)}."
            )
        return self.table[:, : x.size(1)].to(dtype=x.dtype, device=x.device)


class CircularTokenEmbedding(nn.Module):
    """Official Informer-style circular Conv1d value embedding."""

    def __init__(self, input_dim: int, d_model: int):
        super().__init__()
        self.projection = nn.Conv1d(
            input_dim,
            d_model,
            kernel_size=3,
            padding=1,
            padding_mode="circular",
            bias=False,
        )
        nn.init.kaiming_normal_(
            self.projection.weight,
            mode="fan_in",
            nonlinearity="leaky_relu",
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.projection(x.transpose(1, 2)).transpose(1, 2)


class InformerDataEmbedding(nn.Module):
    """Value + position embedding; UCR has no calendar/time-marker covariates."""

    def __init__(self, input_dim: int, d_model: int, dropout: float):
        super().__init__()
        self.value_embedding = CircularTokenEmbedding(input_dim, d_model)
        self.position_embedding = SinusoidalPositionalEmbedding(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        value = self.value_embedding(x)
        return self.dropout(value + self.position_embedding(value))


class ProbSparseSelfAttention(nn.Module):
    """Non-causal ProbSparse self-attention for an Informer encoder.

    Queries are ranked using sampled key interactions. Only the selected top
    queries receive exact attention over every key; remaining queries use the
    non-causal mean-value context used by Informer's encoder formulation.
    """

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        factor: int = 5,
        attention_dropout: float = 0.1,
    ):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads.")
        if factor < 1:
            raise ValueError("ProbSparse factor must be at least 1.")
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.factor = int(factor)
        self.scale = self.head_dim ** -0.5
        self.query_projection = nn.Linear(d_model, d_model)
        self.key_projection = nn.Linear(d_model, d_model)
        self.value_projection = nn.Linear(d_model, d_model)
        self.output_projection = nn.Linear(d_model, d_model)
        self.attention_dropout = nn.Dropout(attention_dropout)

    @staticmethod
    def _log_sample_count(length: int, factor: int) -> int:
        if length <= 1:
            return 1
        return min(length, max(1, int(factor * math.ceil(math.log(length)))))

    def _shape_heads(self, x: torch.Tensor) -> torch.Tensor:
        batch, length, _ = x.shape
        return x.view(batch, length, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, query_length, _ = x.shape
        key_length = query_length
        queries = self._shape_heads(self.query_projection(x))
        keys = self._shape_heads(self.key_projection(x))
        values = self._shape_heads(self.value_projection(x))

        sample_k = self._log_sample_count(key_length, self.factor)
        n_top = self._log_sample_count(query_length, self.factor)

        # One sampled key set per query position; shared over batch and heads,
        # matching the stochastic sparse-query selection idea in Informer.
        sampled_key_index = torch.randint(
            key_length,
            (query_length, sample_k),
            device=x.device,
        )
        sampled_keys = keys[:, :, sampled_key_index, :]
        sampled_scores = torch.matmul(
            queries.unsqueeze(-2),
            sampled_keys.transpose(-2, -1),
        ).squeeze(-2)
        # Keep the official Informer sparsity estimator: sampled-score maximum
        # minus the sampled-score sum normalized by the full key length.
        sparsity = (
            sampled_scores.max(dim=-1).values
            - sampled_scores.sum(dim=-1) / key_length
        )
        top_query_index = sparsity.topk(n_top, dim=-1, sorted=False).indices

        gather_index = top_query_index.unsqueeze(-1).expand(
            batch,
            self.n_heads,
            n_top,
            self.head_dim,
        )
        top_queries = queries.gather(dim=2, index=gather_index)
        exact_scores = torch.matmul(top_queries, keys.transpose(-2, -1)) * self.scale
        # The reference ProbAttention path applies softmax directly here. The
        # attention-dropout argument is retained for interface/config parity.
        attention = torch.softmax(exact_scores, dim=-1)
        top_context = torch.matmul(attention, values)

        context = values.mean(dim=2, keepdim=True).expand(
            batch,
            self.n_heads,
            query_length,
            self.head_dim,
        ).clone()
        context.scatter_(dim=2, index=gather_index, src=top_context)
        context = context.transpose(1, 2).contiguous().view(batch, query_length, -1)
        return self.output_projection(context)


class OfficialInformerEncoderLayer(nn.Module):
    """ProbSparse residual block with the official 1x1 Conv feed-forward form."""

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        d_ff: int,
        factor: int,
        dropout: float,
        attention_dropout: float,
        activation: str,
    ):
        super().__init__()
        self.attention = ProbSparseSelfAttention(
            d_model=d_model,
            n_heads=n_heads,
            factor=factor,
            attention_dropout=attention_dropout,
        )
        self.conv1 = nn.Conv1d(d_model, d_ff, kernel_size=1)
        self.conv2 = nn.Conv1d(d_ff, d_model, kernel_size=1)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU() if activation.lower() == "gelu" else nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attended = self.attention(x)
        x = x + self.dropout(attended)
        residual = x = self.norm1(x)
        feed_forward = self.conv1(x.transpose(1, 2))
        feed_forward = self.dropout(self.activation(feed_forward))
        feed_forward = self.dropout(self.conv2(feed_forward).transpose(1, 2))
        return self.norm2(residual + feed_forward)


class OfficialInformerDistilLayer(nn.Module):
    """Official Conv1d–BatchNorm–ELU–MaxPool sequence distillation."""

    def __init__(self, d_model: int):
        super().__init__()
        self.down_conv = nn.Conv1d(
            d_model,
            d_model,
            kernel_size=3,
            padding=1,
            padding_mode="circular",
        )
        self.norm = nn.BatchNorm1d(d_model)
        self.activation = nn.ELU()
        self.pool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.down_conv(x.transpose(1, 2))
        x = self.norm(x)
        x = self.activation(x)
        x = self.pool(x)
        return x.transpose(1, 2)


class InformerProbSparseEncoder(nn.Module):
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        e_layers: int,
        d_ff: int,
        factor: int,
        dropout: float,
        attention_dropout: float,
        distil: bool,
        activation: str,
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                OfficialInformerEncoderLayer(
                    d_model=d_model,
                    n_heads=n_heads,
                    d_ff=d_ff,
                    factor=factor,
                    dropout=dropout,
                    attention_dropout=attention_dropout,
                    activation=activation,
                )
                for _ in range(e_layers)
            ]
        )
        self.distil_layers = (
            nn.ModuleList(
                [OfficialInformerDistilLayer(d_model) for _ in range(e_layers - 1)]
            )
            if distil and e_layers > 1
            else None
        )
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for index, layer in enumerate(self.layers):
            x = layer(x)
            if self.distil_layers is not None and index < len(self.distil_layers):
                x = self.distil_layers[index](x)
        return self.final_norm(x)


class TimePoolClassifierHead(nn.Module):
    def __init__(self, d_model: int, num_outputs: int, pool: str, dropout: float):
        super().__init__()
        if pool not in {"mean", "max"}:
            raise ValueError("Pooling must be 'mean' or 'max'.")
        self.pool = pool
        self.dropout = nn.Dropout(dropout)
        self.projection = nn.Linear(d_model, num_outputs)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pooled = x.mean(dim=1) if self.pool == "mean" else x.max(dim=1).values
        return self.projection(self.dropout(pooled))


class InformerProbSparseClassifier(nn.Module):
    """Official-core Informer encoder with a UCR classification head."""

    def __init__(
        self,
        input_dim: int,
        num_outputs: int,
        d_model: int = 256,
        n_heads: int = 4,
        e_layers: int = 2,
        d_ff: int = 512,
        factor: int = 5,
        dropout: float = 0.1,
        attention_dropout: float = 0.1,
        distil: bool = True,
        activation: str = "gelu",
        pool: str = "mean",
    ):
        super().__init__()
        self.embedding = InformerDataEmbedding(input_dim, d_model, dropout)
        self.encoder = InformerProbSparseEncoder(
            d_model=d_model,
            n_heads=n_heads,
            e_layers=e_layers,
            d_ff=d_ff,
            factor=factor,
            dropout=dropout,
            attention_dropout=attention_dropout,
            distil=distil,
            activation=activation,
        )
        self.head = TimePoolClassifierHead(d_model, num_outputs, pool, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim == 2:
            x = x.unsqueeze(-1)
        return self.head(self.encoder(self.embedding(x)))


class TimeSeriesDataset(Dataset):
    def __init__(self, time_series: np.ndarray, labels: np.ndarray):
        if len(time_series) != len(labels):
            raise ValueError("Time-series and label sample counts are not aligned.")
        self.time_series = time_series.astype("float32")
        self.labels = labels.astype("float32")

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int):
        return self.time_series[index], self.labels[index]


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


In [ ]:
# Cell 5 - common loss evaluation and corrected binary/multiclass metrics
def evaluate_loss(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    """Sample-weighted mean BCE loss. Does not compute or inspect test metrics."""
    model.eval()
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=torch.float32, non_blocking=True)
            y = y.to(device=device, dtype=torch.float32, non_blocking=True)
            logits = model(x)
            loss = criterion(logits, y)
            batch_n = int(y.size(0))
            total_loss += float(loss.item()) * batch_n
            total_samples += batch_n
    if total_samples == 0:
        raise ValueError("Cannot evaluate an empty loader.")
    return total_loss / total_samples


def macro_ovr_auc_from_columns(
    y_true_columns: np.ndarray,
    probabilities: np.ndarray,
) -> float:
    class_aucs = []
    for column in range(y_true_columns.shape[1]):
        y_column = y_true_columns[:, column]
        if np.unique(y_column).size < 2:
            continue
        class_aucs.append(roc_auc_score(y_column, probabilities[:, column]))
    return float(np.mean(class_aucs)) if class_aucs else float("nan")


def corrected_classification_metrics(
    logits: np.ndarray,
    labels: np.ndarray,
) -> tuple[float, float]:
    logits = np.asarray(logits)
    labels = np.asarray(labels)
    if logits.ndim == 1:
        logits = logits[:, None]
    if labels.ndim == 1:
        labels = labels[:, None]
    if logits.shape != labels.shape:
        raise ValueError(f"Logit/label shape mismatch: {logits.shape} vs {labels.shape}")

    probabilities = 1.0 / (1.0 + np.exp(-np.clip(logits, -50.0, 50.0)))
    if logits.shape[1] == 1:
        y_true = labels[:, 0].astype(np.int64)
        y_pred = (probabilities[:, 0] >= 0.5).astype(np.int64)
        accuracy = accuracy_score(y_true, y_pred)
        auc = (
            roc_auc_score(y_true, probabilities[:, 0])
            if np.unique(y_true).size == 2
            else float("nan")
        )
    else:
        y_true = np.argmax(labels, axis=1)
        y_pred = np.argmax(logits, axis=1)
        accuracy = accuracy_score(y_true, y_pred)
        auc = macro_ovr_auc_from_columns(labels, probabilities)
    return float(auc), float(accuracy)


def evaluate_test_once(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float, float, int]:
    model.eval()
    logits_parts, label_parts = [], []
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=torch.float32, non_blocking=True)
            y_device = y.to(device=device, dtype=torch.float32, non_blocking=True)
            logits = model(x)
            loss = criterion(logits, y_device)
            batch_n = int(y.size(0))
            total_loss += float(loss.item()) * batch_n
            total_samples += batch_n
            logits_parts.append(logits.detach().cpu().numpy())
            label_parts.append(y.numpy())

    if total_samples == 0:
        raise ValueError("Cannot evaluate an empty TEST loader.")
    logits_np = np.concatenate(logits_parts, axis=0)
    labels_np = np.concatenate(label_parts, axis=0)
    test_auc, test_acc = corrected_classification_metrics(logits_np, labels_np)
    return test_auc, test_acc, float(total_loss / total_samples), int(total_samples)


In [ ]:
# Cell 6 - fast internal self-checks (no UCR files required)
def run_internal_self_checks() -> None:
    binary_logits = np.asarray([[-2.0], [2.0], [1.0], [-1.0]], dtype=np.float32)
    binary_labels = np.asarray([[0.0], [1.0], [0.0], [1.0]], dtype=np.float32)
    binary_auc, binary_acc = corrected_classification_metrics(binary_logits, binary_labels)
    if not np.isclose(binary_acc, 0.5):
        raise AssertionError(f"Binary metric self-check failed: ACC={binary_acc}")

    multiclass_logits = np.asarray(
        [[4.0, -1.0, -2.0], [-1.0, 3.0, 0.0], [0.0, -1.0, 3.0]],
        dtype=np.float32,
    )
    multiclass_labels = np.eye(3, dtype=np.float32)
    _, multiclass_acc = corrected_classification_metrics(
        multiclass_logits,
        multiclass_labels,
    )
    if not np.isclose(multiclass_acc, 1.0):
        raise AssertionError("Multiclass metric self-check failed.")

    cpu_model = InformerProbSparseClassifier(
        input_dim=1,
        num_outputs=1,
        d_model=16,
        n_heads=2,
        e_layers=2,
        factor=5,
        d_ff=32,
        dropout=0.0,
        attention_dropout=0.0,
        distil=True,
        activation="gelu",
        pool="mean",
    ).cpu()
    cpu_model.eval()
    with torch.no_grad():
        output = cpu_model(torch.randn(2, 8, 1))
    if tuple(output.shape) != (2, 1):
        raise AssertionError(f"Model forward shape is wrong: {tuple(output.shape)}")

    print(
        "INTERNAL SELF-CHECKS PASSED | "
        f"binary ACC test={binary_acc:.2f} (expected 0.50, not forced to 1.00) | "
        f"binary AUC test={binary_auc:.2f} | ProbSparse model output={tuple(output.shape)}"
    )


run_internal_self_checks()


In [ ]:
# Cell 7 - one complete official-core ProbSparse Informer dataset run
def run_one_dataset_v2(
    dataset_dir: Path,
    dataset_name: str,
    output_paths: dict[str, Path],
    device: torch.device = DEVICE,
    batch_size: int = BATCH_SIZE,
    num_epochs: int = NUM_EPOCHS,
    lr: float = LEARNING_RATE,
    val_fraction: float = VAL_FRACTION,
    save_curve: bool = SAVE_INDIVIDUAL_CURVES,
) -> tuple[dict, pd.DataFrame]:
    set_global_seed(SEED)
    if device.type == "cuda":
        torch.cuda.empty_cache()

    required_paths = {
        "train_tsv": dataset_dir / f"{dataset_name}_TRAIN_cleaned.tsv",
        "test_tsv": dataset_dir / f"{dataset_name}_TEST_cleaned.tsv",
    }
    missing = [str(path) for path in required_paths.values() if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing required files: " + "; ".join(missing))

    # Official TRAIN and TEST are never concatenated. Informer does not read Aout.
    tsv_train = pd.read_csv(required_paths["train_tsv"], sep="\t", header=None)
    tsv_test = pd.read_csv(required_paths["test_tsv"], sep="\t", header=None)

    y_train_raw = tsv_train.iloc[:, 0].to_numpy()
    y_test_raw = tsv_test.iloc[:, 0].to_numpy()
    time_train_raw = tsv_train.iloc[:, 1:].to_numpy(dtype="float32")
    time_test_raw = tsv_test.iloc[:, 1:].to_numpy(dtype="float32")

    temporary_train, _, _ = clean_and_pad_timeseries(
        time_train_raw,
        min_len=MIN_EFFECTIVE_LENGTH,
        fixed_len=None,
    )
    train_sequence_length = int(temporary_train.shape[1])

    time_train_clean, keep_train, _ = clean_and_pad_timeseries(
        time_train_raw,
        min_len=MIN_EFFECTIVE_LENGTH,
        fixed_len=train_sequence_length,
    )
    time_test_clean, keep_test, _ = clean_and_pad_timeseries(
        time_test_raw,
        min_len=MIN_EFFECTIVE_LENGTH,
        fixed_len=train_sequence_length,
    )
    y_train = y_train_raw[keep_train]
    y_test = y_test_raw[keep_test]

    classes = np.sort(np.unique(y_train))
    if len(classes) < 2:
        raise ValueError("Training split contains fewer than two classes.")
    unseen_test = np.setdiff1d(np.unique(y_test), classes)
    if len(unseen_test):
        raise ValueError(f"Official TEST contains labels absent from TRAIN: {unseen_test}")

    y_train_encoded = label_binarize(y_train, classes=classes).astype("float32")
    y_test_encoded = label_binarize(y_test, classes=classes).astype("float32")
    if y_train_encoded.ndim == 1:
        y_train_encoded = y_train_encoded[:, None]
        y_test_encoded = y_test_encoded[:, None]

    train_idx, val_idx, split_path = make_class_safe_split(
        labels=y_train,
        dataset_name=dataset_name,
        split_dir=output_paths["shared_splits"],
        val_fraction=val_fraction,
        seed=SEED,
    )

    time_train = time_train_clean[:, :, None].astype("float32")
    time_test = time_test_clean[:, :, None].astype("float32")
    train_dataset = TimeSeriesDataset(
        time_train[train_idx],
        y_train_encoded[train_idx],
    )
    val_dataset = TimeSeriesDataset(
        time_train[val_idx],
        y_train_encoded[val_idx],
    )
    test_dataset = TimeSeriesDataset(time_test, y_test_encoded)

    generator = torch.Generator(device="cpu").manual_seed(SEED)
    pin_memory = device.type == "cuda"
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        pin_memory=pin_memory,
        num_workers=NUM_WORKERS,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_memory,
        num_workers=NUM_WORKERS,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_memory,
        num_workers=NUM_WORKERS,
    )

    num_outputs = int(y_train_encoded.shape[1])
    model = InformerProbSparseClassifier(
        input_dim=int(time_train.shape[2]),
        num_outputs=num_outputs,
        d_model=INFORMER_D_MODEL,
        n_heads=INFORMER_N_HEADS,
        e_layers=INFORMER_E_LAYERS,
        factor=INFORMER_FACTOR,
        d_ff=INFORMER_D_FF,
        dropout=INFORMER_DROPOUT,
        attention_dropout=INFORMER_ATTN_DROPOUT,
        distil=INFORMER_DISTIL,
        activation=INFORMER_ACTIVATION,
        pool=INFORMER_POOL,
    ).to(device)

    parameter_count = count_trainable_parameters(model)
    print(
        f"[{dataset_name}] model={model.__class__.__name__} | "
        f"uses_aout=False | attention=ProbSparse | distil={INFORMER_DISTIL} | "
        f"outputs={num_outputs} | "
        f"classes={len(classes)} | params={parameter_count:,} | train/val/test="
        f"{len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}"
    )

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=T_MAX, eta_min=ETA_MIN)

    checkpoint_path = output_paths["checkpoints"] / f"{dataset_name}_best.pt"
    best_val_loss = float("inf")
    best_epoch = 0
    history_rows = []

    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_samples = 0
        for x, y in train_loader:
            x = x.to(device=device, dtype=torch.float32, non_blocking=True)
            y = y.to(device=device, dtype=torch.float32, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            batch_n = int(y.size(0))
            train_loss_sum += float(loss.item()) * batch_n
            train_samples += batch_n

        train_loss = train_loss_sum / train_samples
        val_loss = evaluate_loss(model, val_loader, criterion, device)
        current_lr = float(optimizer.param_groups[0]["lr"])
        history_rows.append(
            {
                "dataset": dataset_name,
                "model": MODEL_NAME,
                "seed": SEED,
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "learning_rate": current_lr,
            }
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            checkpoint = {
                "experiment_version": EXPERIMENT_VERSION,
                "dataset": dataset_name,
                "model_name": MODEL_NAME,
                "model_class": model.__class__.__name__,
                "state_dict": copy.deepcopy(model.state_dict()),
                "best_epoch": best_epoch,
                "best_val_loss": best_val_loss,
                "classes": classes.tolist(),
                "parameter_count": parameter_count,
                "train_sequence_length": train_sequence_length,
                "split_path": str(split_path),
                "architecture": {
                    "d_model": INFORMER_D_MODEL,
                    "n_heads": INFORMER_N_HEADS,
                    "factor": INFORMER_FACTOR,
                    "e_layers": INFORMER_E_LAYERS,
                    "d_ff": INFORMER_D_FF,
                    "dropout": INFORMER_DROPOUT,
                    "attn_dropout": INFORMER_ATTN_DROPOUT,
                    "distil": INFORMER_DISTIL,
                    "activation": INFORMER_ACTIVATION,
                    "pool": INFORMER_POOL,
                },
            }
            atomic_torch_save(checkpoint, checkpoint_path)

        scheduler.step()

    history_df = pd.DataFrame(history_rows)
    atomic_write_csv(
        history_df,
        output_paths["dataset_histories"] / f"{dataset_name}.csv",
    )

    checkpoint = load_own_checkpoint(checkpoint_path, device)
    model.load_state_dict(checkpoint["state_dict"])
    # ProbSparse samples keys. Reset only before the single official TEST
    # evaluation so the reported inference is deterministic and does not
    # influence training or checkpoint selection.
    set_global_seed(SEED)
    test_auc, test_acc, test_loss, test_n = evaluate_test_once(
        model,
        test_loader,
        criterion,
        device,
    )

    if save_curve:
        figure, axis = plt.subplots(figsize=(6.2, 4.0))
        axis.plot(history_df["epoch"], history_df["train_loss"], label="Train BCE")
        axis.plot(history_df["epoch"], history_df["val_loss"], label="Validation BCE")
        axis.axvline(
            best_epoch,
            color="black",
            linestyle="--",
            linewidth=1,
            label="Best epoch",
        )
        axis.set(
            title=f"{dataset_name} - Informer ProbSparse V2",
            xlabel="Epoch",
            ylabel="BCE loss",
        )
        axis.grid(alpha=0.25)
        axis.legend()
        figure.tight_layout()
        figure.savefig(output_paths["curves"] / f"{dataset_name}_loss.png", dpi=180)
        plt.close(figure)

    result = {
        "dataset": dataset_name,
        "model": MODEL_NAME,
        "seed": SEED,
        "test_auc": test_auc,
        "test_acc": test_acc,
        "test_loss": test_loss,
        "n_samples": test_n,
        "num_classes": int(len(classes)),
        "num_outputs": num_outputs,
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss),
        "train_samples": int(len(train_dataset)),
        "val_samples": int(len(val_dataset)),
        "parameter_count": int(parameter_count),
        "uses_aout": False,
        "attention": "ProbSparse",
        "probsparse_factor": int(INFORMER_FACTOR),
        "distillation": bool(INFORMER_DISTIL),
        "status": "completed",
    }
    print(
        f"[{dataset_name}] TEST loss={test_loss:.4f} | AUC={test_auc:.4f} | "
        f"ACC={test_acc:.4f} | best_epoch={best_epoch} | n={test_n}"
    )
    return result, history_df


In [ ]:
# Cell 8 - discovery, resumable all-dataset runner, summaries, and persistence
def discover_datasets(root: Path) -> list[str]:
    """
    Use the same dataset discovery rule as Two-Tower V2 for a matched cohort.

    Aout files are marker files only here; Informer never reads their values.
    """
    if not root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {root}")
    names = []
    for subdir in sorted(path for path in root.iterdir() if path.is_dir()):
        name = subdir.name
        required = [
            subdir / f"{name}_TRAIN_cleaned.tsv",
            subdir / f"{name}_TEST_cleaned.tsv",
            subdir / f"{name}_Aout_train_k2.csv",
            subdir / f"{name}_Aout_test_k2.csv",
        ]
        if all(path.exists() for path in required):
            names.append(name)
    return names


def summarize_results(results_df: pd.DataFrame) -> dict:
    valid_auc = results_df.dropna(subset=["test_auc"])
    total_n = float(results_df["n_samples"].sum())
    auc_weight_n = float(valid_auc["n_samples"].sum())
    return {
        "datasets_completed": int(len(results_df)),
        "simple_mean_auc": float(valid_auc["test_auc"].mean()),
        "simple_mean_acc": float(results_df["test_acc"].mean()),
        "simple_mean_loss": float(results_df["test_loss"].mean()),
        "weighted_auc": float(
            (valid_auc["test_auc"] * valid_auc["n_samples"]).sum() / auc_weight_n
        ),
        "weighted_acc": float(
            (results_df["test_acc"] * results_df["n_samples"]).sum() / total_n
        ),
    }


def run_datasets_v2(
    root: Path,
    selected: list[str] | None = None,
    debug: bool = False,
    num_epochs: int = NUM_EPOCHS,
    resume: bool = RESUME_COMPLETED_DATASETS,
) -> tuple[pd.DataFrame, dict]:
    base_dir = model_output_dir(debug=debug)
    output_paths = ensure_output_tree(base_dir)
    final_results_path = base_dir / "final_results.csv"
    histories_path = base_dir / "histories.csv"
    failed_path = base_dir / "failed_datasets.csv"
    config_path = base_dir / "run_config.json"
    summary_path = base_dir / "summary.json"

    atomic_write_json(base_run_config(debug=debug, epochs=num_epochs), config_path)

    available = discover_datasets(root)
    if selected is None:
        dataset_names = available
    else:
        missing = sorted(set(selected) - set(available))
        if missing:
            raise FileNotFoundError(f"Selected datasets not discoverable: {missing}")
        dataset_names = [name for name in selected if name in available]

    completed = set()
    if resume and final_results_path.exists():
        existing = pd.read_csv(final_results_path)
        if "dataset" in existing.columns and "status" in existing.columns:
            completed = set(existing.loc[existing["status"] == "completed", "dataset"])

    print(f"Output directory: {base_dir}")
    print(f"Datasets requested: {len(dataset_names)}")
    print(f"Already completed and skipped: {len(completed.intersection(dataset_names))}")

    for position, dataset_name in enumerate(dataset_names, start=1):
        if dataset_name in completed:
            print(f"[{position}/{len(dataset_names)}] Skip completed: {dataset_name}")
            continue
        print(f"\n[{position}/{len(dataset_names)}] Start: {dataset_name}")
        try:
            result, history_df = run_one_dataset_v2(
                dataset_dir=root / dataset_name,
                dataset_name=dataset_name,
                output_paths=output_paths,
                device=DEVICE,
                batch_size=BATCH_SIZE,
                num_epochs=num_epochs,
                lr=LEARNING_RATE,
                val_fraction=VAL_FRACTION,
                save_curve=SAVE_INDIVIDUAL_CURVES,
            )
            append_or_replace_dataset_rows(
                final_results_path,
                pd.DataFrame([result]),
                dataset_name,
            )
            append_or_replace_dataset_rows(histories_path, history_df, dataset_name)

            if failed_path.exists():
                failed_df = pd.read_csv(failed_path)
                failed_df = failed_df[failed_df["dataset"] != dataset_name]
                atomic_write_csv(failed_df, failed_path)
        except Exception as error:
            failure = pd.DataFrame(
                [
                    {
                        "dataset": dataset_name,
                        "model": MODEL_NAME,
                        "seed": SEED,
                        "error_type": type(error).__name__,
                        "error_message": str(error),
                        "traceback": traceback.format_exc(),
                        "recorded_utc": datetime.now(timezone.utc).isoformat(),
                    }
                ]
            )
            append_or_replace_dataset_rows(failed_path, failure, dataset_name)
            print(f"FAILED {dataset_name}: {type(error).__name__}: {error}")

    if not final_results_path.exists():
        raise RuntimeError("No dataset completed successfully.")

    final_results = pd.read_csv(final_results_path).sort_values("dataset").reset_index(drop=True)
    requested_results = final_results[final_results["dataset"].isin(dataset_names)].copy()
    summary = summarize_results(requested_results)
    summary.update(
        {
            "model": MODEL_NAME,
            "seed": SEED,
            "debug": bool(debug),
            "datasets_requested": int(len(dataset_names)),
            "generated_utc": datetime.now(timezone.utc).isoformat(),
        }
    )
    atomic_write_json(summary, summary_path)

    print("\n========== V2 SUMMARY (OFFICIAL TEST, EVALUATED ONCE) ==========")
    print(
        requested_results[
            ["dataset", "test_auc", "test_acc", "test_loss", "n_samples"]
        ].to_string(index=False)
    )
    print(
        f"\nSimple mean: AUC={summary['simple_mean_auc']:.4f}, "
        f"ACC={summary['simple_mean_acc']:.4f}, LOSS={summary['simple_mean_loss']:.4f}"
    )
    print(
        f"Weighted: AUC={summary['weighted_auc']:.4f}, "
        f"ACC={summary['weighted_acc']:.4f}"
    )
    return requested_results, summary


## Smoke test

The smoke test uses one binary and one multiclass dataset for three epochs. It
checks file discovery, shared splitting, output shapes, corrected metrics,
checkpoint restoration, and persistence. Its scores are not paper results.


In [ ]:
# Cell 9 - safe three-epoch smoke test (enabled by default)
if RUN_SMOKE_TEST:
    smoke_results, smoke_summary = run_datasets_v2(
        root=DATA_ROOT,
        selected=SMOKE_DATASETS,
        debug=True,
        num_epochs=SMOKE_EPOCHS,
        resume=False,
    )
else:
    print("Smoke test disabled. Set RUN_SMOKE_TEST=True in Cell 1 to run it.")


## Final 125-dataset run

Run this only after the smoke test completes. Set `RUN_SMOKE_TEST=False` and
`RUN_FULL_EXPERIMENT=True`, restart the kernel, clear outputs, and run all cells.
The same 60-epoch budget and shared validation splits are used for every model.


In [ ]:
# Cell 10 - final full run (disabled by default)
if RUN_FULL_EXPERIMENT:
    full_results, full_summary = run_datasets_v2(
        root=DATA_ROOT,
        selected=None,
        debug=False,
        num_epochs=NUM_EPOCHS,
        resume=RESUME_COMPLETED_DATASETS,
    )
else:
    print("Full run disabled. Set RUN_FULL_EXPERIMENT=True in Cell 1 after smoke testing.")


## Files produced by the final run

Under `final_runs_v2/informer_probsparse/seed_42/`:

- `final_results.csv`: one official TEST result per completed dataset;
- `histories.csv`: all datasets' 60-epoch train/validation BCE histories;
- `dataset_histories/<dataset>.csv`: one history file per dataset;
- `checkpoints/<dataset>_best.pt`: minimum-validation-loss checkpoint;
- `curves/<dataset>_loss.png`: train/validation BCE curve and best epoch;
- `failed_datasets.csv`: generic failure records, including tracebacks;
- `run_config.json`: common protocol plus ProbSparse-specific configuration;
- `summary.json`: simple and sample-weighted aggregate metrics.

The six representative datasets used elsewhere in the paper can later be
selected directly from `final_results.csv` and `histories.csv`; no retraining is
needed for metric tables or train/validation-loss figures.

For probability-level ROC curves or confusion matrices, use each saved best
checkpoint to make one deterministic inference pass and save per-sample outputs.
